# Semantics of domains, values and fields

Computing a new field from other fields means computing two things:

- its **domain**: the points where the new field is defined, and
- its **values**: the value of the new field at each of those points.

Each has its own choice of strategy, and the two choices are independent:

| | Options |
|---|---|
| **Domain computation** | *forward*: from the input domains to the largest output domain; *backward*: from the required output domain to the smallest input domains |
| **Value computation** | *operation by operation*: every intermediate field computed on its whole domain; *point by point*: all intermediate values for one point before the next point |

This notebook explains both choices with tiny 1D examples in plain Python and numpy. It then maps the choices to GT4Py's field view and iterator view, and shows how a third choice, eager or lazy evaluation, relates to them. Along the way, each strategy is matched to a functional-programming pattern: functor, comonad and Coyoneda.

> **Terminology.** A field `f` is a function from its domain to its codomain.
> - **Domain**: the set of points where `f` is defined, e.g. `range(0, 10)`.
> - **Codomain**: the set its values belong to, e.g. `float64`. In `gt4py.next`, this is what `Field.codomain` returns.
> - **Values**: `f(x)` for each point `x` of the domain.
> - **Graph**: the set of pairs `(x, f(x))`. The graph fully describes the domain and the values.

## 1. Fields

A field maps every point of its domain to a value. In these examples, a 1D field is a pair `(domain, values)`: the domain is a `range` of indices, and the values are a numpy array with one entry per point. Pairing the two gives the field's graph: the point `domain[k]` has the value `values[k]`.

In [1]:
import numpy as np

# A 1D field: a domain (a range of indices) and its values on that domain.
a = (range(0, 10), np.arange(10.0) ** 2)  # a(i) = i**2 for i in 0..9


def value(field, i):
    """The value of `field` at point `i`."""
    domain, values = field
    return values[i - domain.start]


def restrict(field, domain):
    """The part of `field` on `domain`."""
    field_domain, values = field
    start = domain.start - field_domain.start
    return (domain, values[start : start + len(domain)])


print(value(a, 3), restrict(a, range(2, 5)))

9.0 (range(2, 5), array([ 4.,  9., 16.]))


## 2. Domain computation: forward or backward

All the examples use the same computation, two stencils applied one after the other:

- `tmp(i) = a(i - 1) + a(i + 1)` reads `a` at offsets `-1` and `+1`;
- `out(i) = tmp(i + 1) - tmp(i)` reads `tmp` at offsets `0` and `+1`.

A stencil can only be computed at points where all its reads fall inside the domain of its input. The range of these offsets, `(min, max)`, is often called the stencil's *extent*. It links input and output domains in both directions:

- **Forward propagation** goes from the input domains to the **largest** output domain that can be computed. If a smaller output domain is needed in the end, the result is sliced afterwards.
- **Backward propagation** goes from the **required** output domain to the **smallest** input domains needed to compute it. The inputs are restricted to those domains first, so nothing is computed that isn't needed. This only works if the required output domain is known before computing.

In [2]:
# Two stencils and the offsets at which they read their input.
#   tmp(i) = a(i - 1) + a(i + 1)
#   out(i) = tmp(i + 1) - tmp(i)
OP1_OFFSETS = (-1, +1)
OP2_OFFSETS = (0, +1)


def forward(input_domain, offsets):
    """The largest output domain whose reads at `offsets` all stay inside `input_domain`."""
    return range(input_domain.start - min(offsets), input_domain.stop - max(offsets))


def backward(output_domain, offsets):
    """The smallest input domain needed to compute `output_domain` reading at `offsets`."""
    return range(output_domain.start + min(offsets), output_domain.stop + max(offsets))


# Forward: from the input domain to the largest possible output domain.
tmp_domain = forward(a[0], OP1_OFFSETS)
out_domain = forward(tmp_domain, OP2_OFFSETS)
print("forward: ", a[0], "->", tmp_domain, "->", out_domain)

# Backward: from the required output domain to the smallest needed input domain.
required = range(3, 5)
tmp_needed = backward(required, OP2_OFFSETS)
a_needed = backward(tmp_needed, OP1_OFFSETS)
print("backward:", a_needed, "<-", tmp_needed, "<-", required)

forward:  range(0, 10) -> range(1, 9) -> range(1, 8)
backward: range(2, 7) <- range(3, 6) <- range(3, 5)


The next cell computes the values for the required domain `range(3, 5)` both ways. Both give the same result. Forward computes 15 points (8 of `tmp` and 7 of `out`) and throws most of them away. Backward computes only the 5 that are needed (3 of `tmp` and 2 of `out`).

In [3]:
def shifted(field, domain, offset):
    """The values of `field` at `i + offset` for every `i` in `domain`, as one numpy slice."""
    return restrict(field, range(domain.start + offset, domain.stop + offset))[1]


def op1(a, domain):
    return (domain, shifted(a, domain, -1) + shifted(a, domain, +1))


def op2(tmp, domain):
    return (domain, shifted(tmp, domain, +1) - shifted(tmp, domain, 0))


# Forward: compute everything that can be computed, then cut out the required domain.
tmp = op1(a, tmp_domain)
out = op2(tmp, out_domain)
print("forward: ", len(tmp[0]) + len(out[0]), "points computed ->", restrict(out, required))

# Backward: restrict the input first, then compute only what is needed.
tmp = op1(restrict(a, a_needed), tmp_needed)
out = op2(tmp, required)
print("backward:", len(tmp[0]) + len(out[0]), "points computed ->", out)

forward:  15 points computed -> (range(3, 5), array([14., 18.]))
backward: 5 points computed -> (range(3, 5), array([14., 18.]))


## 3. Value computation: always forward, in one of two orders

Values can only be computed **forward**: `out` needs `tmp`, and `tmp` needs `a`. Backward propagation only finds the domains backwards. Once they are known, the values still flow from the inputs to the outputs.

What can change is the **order** in which the values of a chain of operations are computed:

- **Operation by operation**: compute every value of the first intermediate field on its whole domain, then every value of the next one, and so on. The previous cell works this way: `tmp` is fully computed before `out`.
- **Point by point**: for one point, compute all the intermediate values it needs and its output value, then move to the next point. Intermediate fields are never stored as a whole.

### Pointwise operations: a functor

If every operation is *pointwise* (it only reads its input at the same point), applying an operation to a field is a **functor map**, `fmap(f, field)`. The two orders then correspond to:

- operation by operation: nested maps, `fmap(h, fmap(g, fmap(f, field)))`;
- point by point: a single map of the composed function, `fmap(h ∘ g ∘ f, field)`.

The functor composition law, `fmap(h) ∘ fmap(g) ∘ fmap(f) == fmap(h ∘ g ∘ f)`, guarantees that both give the same values. The only differences are how many intermediate arrays are stored and how many passes over memory are made.

Stencils are not pointwise, so `fmap` cannot describe them; they need the richer structure of a *comonad*, introduced after the next cell.

In [4]:
def fmap(function, values):
    """Apply `function` to every value (the functor's map, for arrays)."""
    return np.array([function(x) for x in values])


def f(x):
    return x + 1


def g(x):
    return 2 * x


def h(x):
    return x**2


# A pointwise operation keeps the domain unchanged, so only the values are mapped.
values = np.arange(5.0)
# One materialized intermediate array per operation.
nested = fmap(h, fmap(g, fmap(f, values)))
# One pass over the values, no intermediate arrays.
fused = fmap(lambda x: h(g(f(x))), values)
print(nested, fused, np.array_equal(nested, fused))

[  4.  16.  36.  64. 100.] [  4.  16.  36.  64. 100.] True


### Stencils: a comonad

A stencil reads its input at *neighbouring* points, so it is not a function of one value but of **a field seen from one point**: a way to get the value at any point, plus a current point, `w = (get, i)`. This structure is the **Store comonad**, and GT4Py's iterator view calls it an *iterator*. Its operations are:

- `extract(w)`: the value at the current point;
- `extend(stencil, w)`: apply `stencil` at every point, giving a new field seen from the same point;
- `shift(w, offset)` (specific to this comonad): the same field, seen from a neighbouring point.

`fmap` is the special case of a stencil that only reads the current point: `fmap(f) == extend(f ∘ extract)`.

The two value orders of `out = op2(op1(a))` are then:

- **operation by operation**: `extend(op2) ∘ store ∘ extend(op1)`, where `store` computes the intermediate field once on its domain and keeps its values;
- **point by point**: `extend(op2 ∘ extend(op1))`, a single `extend` of the composed stencil, where the inner `extend` computes `op1` on demand wherever `op2` reads it.

The comonad law `extend(f) ∘ extend(g) == extend(f ∘ extend(g))` guarantees that both give the same values (`store` doesn't change any value on its domain): it is the stencil version of the functor composition law. Unlike the pointwise case, the right-hand side **recomputes** `g` at every point where `f` reads it. `op2` reads `tmp` at `i` and `i + 1`, so point by point computes `tmp(i + 1)` once for output point `i` and again for output point `i + 1`.

`store` changes no value: semantically it is the identity (an `fmap(id)`) restricted to its domain. What it changes is the representation, from a getter that recomputes on every read to an array, so that values are computed once and shared. Since `extend` here is lazy, `store` is what tells the two orders apart: without it, `extend(op2) ∘ extend(op1)` recomputes `op1` at every read, exactly like the fused version. The law speaks about values only; the cost depends on where values are stored.

In [5]:
# A field seen from one point, the Store comonad: how to get the value at any point, plus
# the current point. In GT4Py iterator view this is an iterator.
def seen_from(field, i):
    """`field` seen from point `i`."""
    return (lambda j: value(field, j), i)


def extract(w):
    """The value at the current point."""
    get, i = w
    return get(i)


def shift(w, offset):
    """The same field, seen from a neighbouring point."""
    get, i = w
    return (get, i + offset)


def extend(stencil, w):
    """Apply `stencil` at every point, lazily: each value is computed when it is read."""
    get, i = w
    return (lambda j: stencil((get, j)), i)


def values_on(w, domain):
    """The values of `w` on `domain`."""
    get, _ = w
    return np.array([get(j) for j in domain])


def store(w, domain):
    """Compute the values of `w` on `domain` once; later reads come from the stored array."""
    values = values_on(w, domain)
    return (lambda j: values[j - domain.start], w[1])


op1_evaluations = 0


def op1_stencil(w):
    global op1_evaluations
    op1_evaluations += 1
    return extract(shift(w, -1)) + extract(shift(w, +1))


def op2_stencil(w):
    return extract(shift(w, +1)) - extract(w)


def op2_of_op1(w):
    """`op2` composed with `extend(op1)`: `op1` runs at every point `op2` reads."""
    return op2_stencil(extend(op1_stencil, w))


# Operation by operation: `extend(op2) ∘ store ∘ extend(op1)`.
op1_evaluations = 0
tmp = store(extend(op1_stencil, seen_from(a, 0)), tmp_needed)
nested = values_on(extend(op2_stencil, tmp), required)
print("operation by operation:", nested, f"op1 evaluated {op1_evaluations} times")

# Point by point: `extend(op2 ∘ extend(op1))`.
op1_evaluations = 0
fused = values_on(extend(op2_of_op1, seen_from(a, 0)), required)
print("point by point:        ", fused, f"op1 evaluated {op1_evaluations} times")

operation by operation: [14. 18.] op1 evaluated 3 times
point by point:         [14. 18.] op1 evaluated 4 times


Point by point, `op1` ran 4 times for 2 output points. Operation by operation on the backward domains it ran 3 times, once per point of `tmp_needed`, but the whole of `tmp` had to be stored. Compilers choose per intermediate: they fuse where recomputing is cheap and store a *temporary* where it is not. A common middle ground is **tiling**: split the domain into blocks and compute operation by operation within each block, so that intermediates only need to be stored for one block at a time. With stencils, each block's intermediates also need a *halo*, the extra points around the block that the next stencil reads, which is recomputed for each block or exchanged with its neighbours.

## 4. The two choices are orthogonal

The domain direction decides *which* points are computed. The value order decides *in which order* their values are computed. Any combination works, and all four give the same values on the required domain:

| | operation by operation | point by point |
|---|---|---|
| **forward domains** | numpy style; GT4Py field view semantics | e.g. one fused loop over the largest domain |
| **backward domains** | lazy backward evaluation (section 6); classic GridTools stencils (operation by operation per block) | GT4Py iterator view within a stencil; GridTools `fn` (GTFN backend) |

In [6]:
def compute(domain_mode, value_mode, a, required):
    """Compute `out` on `required` with the chosen domain and value strategies."""
    if domain_mode == "forward":
        a_domain = a[0]
        tmp_domain = forward(a_domain, OP1_OFFSETS)
        out_domain = forward(tmp_domain, OP2_OFFSETS)
    else:
        out_domain = required
        tmp_domain = backward(out_domain, OP2_OFFSETS)
        a_domain = backward(tmp_domain, OP1_OFFSETS)
    a = restrict(a, a_domain)

    if value_mode == "nested":
        out = op2(op1(a, tmp_domain), out_domain)
    else:
        out = (out_domain, values_on(extend(op2_of_op1, seen_from(a, 0)), out_domain))
    return restrict(out, required)


for domain_mode in ("forward", "backward"):
    for value_mode in ("nested", "fused"):
        result = compute(domain_mode, value_mode, a, required)
        print(f"{domain_mode:8} + {value_mode:6}: {result[1]}")

forward  + nested: [14. 18.]
forward  + fused : [14. 18.]
backward + nested: [14. 18.]
backward + fused : [14. 18.]


## 5. GT4Py: field view and iterator view

**Field view** (`@field_operator`) works on whole fields. Every expression produces a new field on the intersection of its shifted input domains, and the next expression uses that field: **forward domains, operation by operation**. Embedded execution runs field-view code exactly like this.

**Iterator view**, within a stencil, is the complement on both counts. It is the comonad of section 3 in GT4Py's own words:

- an *iterator* is a field seen from one point;
- `deref` is `extract`, and `shift` moves the iterator;
- `lift(stencil)` is the lazy `extend`: the lifted stencil's values are computed where they are read;
- `as_fieldop(stencil, domain)` is `extend` followed by storing the values on the given domain.

The stencil is applied on a given output domain: **backward domains**. Inside a stencil, `lift` lets it call another stencil, which is recomputed at every point where it is read: **point by point**. Between separate `as_fieldop` calls, though, the values are passed as stored fields, i.e. operation by operation, unless the calls are fused.

The next cell defines these builtins with the comonad operations and computes `out` both ways.

In [7]:
# The iterator builtins, defined with the comonad operations above (`shift` is the same).
# Simplified signatures: GT4Py's `shift` is curried, `shift(offset)(it)`, its `lift` takes
# any number of iterators, and its `as_fieldop` returns a field, not an iterator.
deref = extract


def lift(stencil):
    """`lift(stencil)(it)` is `extend(stencil, it)`: values computed where they are read."""
    return lambda it: extend(stencil, it)


def as_fieldop(stencil, domain):
    """Apply `stencil` on `domain` and store the values: `store(extend(stencil, it), domain)`."""
    return lambda it: store(extend(stencil, it), domain)


def composed(it):
    return op2_stencil(lift(op1_stencil)(it))


it_a = seen_from(a, 0)
# One `as_fieldop` of the composed stencil: point by point.
print(values_on(as_fieldop(composed, required)(it_a), required))
# Two nested `as_fieldop` calls: operation by operation.
tmp_field = as_fieldop(op1_stencil, tmp_needed)(it_a)
print(values_on(as_fieldop(op2_stencil, required)(tmp_field), required))

[14. 18.]
[14. 18.]


### Compiled field view

"Field view = forward + operation by operation" describes the **semantics** of the programming model: what a field-view program means, and how embedded execution runs it. It does not describe how compiled code runs. A compiled field-view program is lowered to an IR of nested `as_fieldop` calls, and the toolchain then switches to the other strategies wherever it pays off:

- `infer_domain` propagates the domains **backward** from the output domains of the program;
- `fuse_as_fieldop` merges nested `as_fieldop` calls into one: **point by point**;
- `global_tmps` stores the intermediates that remain unfused as temporaries: **operation by operation** again, for those intermediates.

These rewrites are allowed precisely because the two choices are orthogonal and all combinations give the same values on the required domain.

## 6. When values are computed: eager or lazy

A third choice, independent of the other two, is **when** the values are computed:

- **Eager**: each operation computes its whole result as soon as it is called, on the largest domain (forward).
- **Lazy**: each operation returns a *deferred* result, whose values are computed when they are first read. Its largest possible domain is already known, since domains follow from the offsets alone, without computing any value.

When matters for the domain direction. Backward propagation needs the required output domain before computing. With `op(a)[domain]`, that domain is only known after the call, so deferring the computation is what makes backward propagation possible there. A lazy evaluator can still compute forward and slice, or propagate the requested domain backward:

| Evaluation | Domain computation | Value computation | Values computed |
|---|---|---|---|
| eager | forward | operation by operation | when the operation is called |
| lazy | forward: computes the largest domain, then slices | operation by operation | when the result is first read |
| lazy backward | backward: restricts each input to the requested domain grown by its offsets | operation by operation | when the result is first read |

If the output domain is passed in the call instead, as with the `out=` and `domain=` arguments of GT4Py, backward propagation needs no deferral. Deferring still pays off for chains of operations: a deferred result passed to another operation is only computed where that operation reads it, so the requested domain propagates backward through the whole chain.

The next cell runs `out = op2(op1(a))` on `required` in the three ways, with the `op1` and `op2` of section 2. All three compute operation by operation; only lazy backward avoids the points that are not needed.

In [8]:
points_computed = 0


def counted(field):
    """Count the points of a computed field."""
    global points_computed
    points_computed += len(field[0])
    return field


class Deferred:
    """`op` applied to `arg`, not computed yet. Its largest possible domain is known."""

    def __init__(self, op, offsets, arg, backward_mode):
        self.op, self.offsets, self.arg, self.backward_mode = op, offsets, arg, backward_mode
        self.domain = forward(domain_of(arg), offsets)  # from the offsets only, no values

    def compute_on(self, domain):
        """Compute the field on `domain`, a part of `self.domain`."""
        if self.backward_mode:  # restrict the input to what `domain` needs
            arg = evaluate(self.arg, backward(domain, self.offsets))
            return counted(self.op(arg, domain))
        arg = evaluate(self.arg, domain_of(self.arg))  # compute everything, then slice
        return restrict(counted(self.op(arg, self.domain)), domain)


def domain_of(x):
    return x.domain if isinstance(x, Deferred) else x[0]


def evaluate(x, domain):
    return x.compute_on(domain) if isinstance(x, Deferred) else restrict(x, domain)


def run(mode, a, required):
    """`out = op2(op1(a))` on `required`, evaluated eagerly or lazily."""
    if mode == "eager":  # each call computes its whole result at once
        tmp = counted(op1(a, forward(a[0], OP1_OFFSETS)))
        out = counted(op2(tmp, forward(tmp[0], OP2_OFFSETS)))
        return restrict(out, required)
    backward_mode = mode == "lazy backward"
    tmp = Deferred(op1, OP1_OFFSETS, a, backward_mode)
    out = Deferred(op2, OP2_OFFSETS, tmp, backward_mode)
    print(f"{mode}: out.domain is {out.domain}, {points_computed} points computed so far")
    return out.compute_on(required)


for mode in ("eager", "lazy", "lazy backward"):
    points_computed = 0
    result = run(mode, a, required)
    print(f"{mode}: {points_computed} points computed -> {result}")

eager: 15 points computed -> (range(3, 5), array([14., 18.]))
lazy: out.domain is range(1, 8), 0 points computed so far
lazy: 15 points computed -> (range(3, 5), array([14., 18.]))
lazy backward: out.domain is range(1, 8), 0 points computed so far
lazy backward: 5 points computed -> (range(3, 5), array([14., 18.]))


### Deferred pointwise operations: Coyoneda

For pointwise operations, a lazy evaluator is a well-known functional pattern, the **Coyoneda** functor: a field together with a *pending* function. Its `fmap` composes the new function after the pending one, without computing any value, and `lower` applies the composed function in a single pass. It turns the nested maps of section 3 into one map of the composed function by construction (the functor law makes the result the same), and it postpones *when* the values are computed until `lower`.

`lower` can also take the requested domain, because restriction commutes with pointwise operations: `restrict(fmap(f, x), D) == fmap(f, restrict(x, D))`. In category-theory terms, restriction is a *natural transformation*. This is backward domain computation for pointwise operations: their extent is zero, so the requested domain is also the input domain needed.

Its dual, the **Yoneda** functor `∀b. (a → b) → F b`, encodes the same deferral in continuation-passing style. For a functor `F`, both are isomorphic to `F a` itself (the Yoneda lemma), so they change *when* the work is done, never the result; Haskell's `kan-extensions` package provides both to fuse chains of `fmap`.

Stencils need more than a pending function: restricting a stencil's result is not the stencil of the restricted input, since the input domain must grow by the stencil's extent (section 2). A lazy evaluator for stencils therefore keeps each operation with its offsets, as `Deferred` does above: an expression to analyse, not only a function to apply.

The next cell applies `f`, `g` and `h` of section 3 to `a` on `required`, eagerly and through `Coyoneda`, counting the calls of the three functions.

In [9]:
class Coyoneda:
    """A field and a pending pointwise function: a deferred `fmap`."""

    def __init__(self, field, function=lambda x: x):
        self.field, self.function = field, function

    def fmap(self, g):
        """Compose `g` after the pending function: no value is computed."""
        f = self.function
        return Coyoneda(self.field, lambda x: g(f(x)))

    def lower(self, domain):
        """Apply the pending function once, at the points of `domain` only."""
        return (domain, fmap(self.function, restrict(self.field, domain)[1]))


calls = 0


def tallied(function):
    """`function`, counting its calls in `calls`."""

    def tallied_function(x):
        global calls
        calls += 1
        return function(x)

    return tallied_function


F, G, H = tallied(f), tallied(g), tallied(h)

# Eager: three passes over the whole field, then restrict.
calls = 0
eager_result = restrict((a[0], fmap(H, fmap(G, fmap(F, a[1])))), required)
print("eager:   ", eager_result, f"{calls} calls")

# Coyoneda: compose the functions, then one pass over the requested domain only.
calls = 0
deferred = Coyoneda(a).fmap(F).fmap(G).fmap(H)
print("deferred:", f"{calls} calls so far")
lowered = deferred.lower(required)
print("lowered: ", lowered, f"{calls} calls")

eager:    (range(3, 5), array([ 400., 1156.])) 30 calls
deferred: 0 calls so far
lowered:  (range(3, 5), array([ 400., 1156.])) 6 calls


## Summary

- A field computation decides **which points** to compute (domain: forward or backward) and **in which order** to compute their values (operation by operation or point by point). Values always flow forward.
- For pointwise operations, the two orders are nested `fmap`s versus one `fmap` of the composed function, equal by the functor law.
- Stencils are comonad `extend`s: the two orders are stored `extend`s versus one `extend` of the composed stencil, equal by the comonad law. Point by point recomputes intermediate values.
- GT4Py field view semantics are forward + operation by operation. Iterator view is the comonad (`deref` = `extract`, `lift` = lazy `extend`): backward, and point by point within a stencil. The compiled toolchain mixes the strategies.
- Eager and lazy evaluation differ in **when** values are computed. Deferring the computation until the required domain is known is what lets a lazy evaluator propagate it backward.
- For pointwise operations, a deferred computation is the Coyoneda functor (a field and a pending function, applied in one pass); restriction commutes with it, which makes backward domains free. Stencils need the deferred expression with its offsets.